# 02 - Baseline models: transaction data vs. client information

This notebook creates a reproducible minimum baseline for the intermediate delivery. It compares two versions of the same labelled purchase data:

- **Transaction-only:** predictors already in `train.csv`;
- **Transaction + client profile:** the same predictors plus selected variables from `clientes.csv`.

`test.csv` is loaded only to confirm that the selected client variables are available for final predictions. It is never used to fit preprocessing, choose features or train a model.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = 'repeat_purchase_90d'

# Works whether Jupyter starts in the repository root or in notebooks/.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'data').exists():
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / 'data'

## 1. Load the official files

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
clientes = pd.read_csv(DATA_DIR / 'clientes.csv')

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
print(f'Clientes: {clientes.shape}')

Train: (6534, 39)
Test: (986, 38)
Clientes: (1143, 13)


## 2. Selected client variables and merge

The selected variables are known at the time of purchase, have manageable cardinality and may capture geographic or customer-cohort patterns. `customer_id` is a join key only and is never used as a predictor.

We deliberately exclude `company_name`, `address` and `street` because they are almost unique per client, and `customer_type` because it is entirely missing. `city` and `postal_code` are postponed to later controlled experiments because they have high cardinality.

In [3]:
CLIENT_FEATURES_CORE = [
    'country_or_region',
    'commercial_region',
    'island',
    'municipality',
    'postal_area',
    'customer_since_year',
]

CLIENT_JOIN_COLUMNS = ['customer_id'] + CLIENT_FEATURES_CORE

assert clientes['customer_id'].is_unique, 'clientes.csv must have one row per customer_id.'
assert train['customer_id'].isin(clientes['customer_id']).all(), 'Some train customers do not match clientes.csv.'
assert test['customer_id'].isin(clientes['customer_id']).all(), 'Some test customers do not match clientes.csv.'

train_with_clients = train.merge(
    clientes[CLIENT_JOIN_COLUMNS],
    on='customer_id',
    how='left',
    validate='many_to_one',
)
test_with_clients = test.merge(
    clientes[CLIENT_JOIN_COLUMNS],
    on='customer_id',
    how='left',
    validate='many_to_one',
)

assert len(train_with_clients) == len(train)
assert len(test_with_clients) == len(test)

print('Safe merge confirmed.')
print('New client predictors:', CLIENT_FEATURES_CORE)
display(train_with_clients[CLIENT_JOIN_COLUMNS + [TARGET]].head())

Safe merge confirmed.
New client predictors: ['country_or_region', 'commercial_region', 'island', 'municipality', 'postal_area', 'customer_since_year']


,customer_id,country_or_region,commercial_region,island,municipality,postal_area,customer_since_year,repeat_purchase_90d
0,660784,Country CT10: Land of The Final Version,Region RG25: We Will See Later,NaN,Municipality MU89: Council of Last Minute Edits,Postal area PA228: Hidden Sheet Sector,2024,1
1,189146,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,Postal area PA260: Hidden Sheet Sector,2015,1
2,103940,Country CT10: Land of The Final Version,Region RG23: We Will See Later,NaN,Municipality MU70: Council of Last Minute Edits,Postal area PA84: Hidden Sheet Sector,2017,1
3,512927,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,Postal area PA253: Hidden Sheet Sector,2018,0
4,122356,Country CT10: Land of The Final Version,Region RG31: We Will See Later,NaN,Municipality MU142: Council of Last Minute Edits,Postal area PA136: Hidden Sheet Sector,2017,1


## 3. Temporal validation split

The supplied test set contains later purchase occasions than the training set. We therefore validate on the latest 20% of dated purchase occasions rather than drawing a random split. Rows with a missing purchase date remain in the training portion; they cannot represent a known future validation period.

This is a first hold-out baseline, not the final assessment strategy. Later we should test the final candidate pipelines with a more robust temporal cross-validation design.

In [4]:
purchase_dates = pd.to_datetime(train['purchase_date'], errors='coerce')
dated_rows = purchase_dates.dropna().sort_values()
cutoff_position = int(len(dated_rows) * 0.80)
cutoff_date = dated_rows.iloc[cutoff_position]

is_validation = purchase_dates.ge(cutoff_date)
is_training = ~is_validation

print(f'Cutoff date: {cutoff_date.date()}')
print(f'Training rows: {is_training.sum():,}')
print(f'Validation rows: {is_validation.sum():,}')
print(f'Training period: {purchase_dates[is_training].min().date()} to {purchase_dates[is_training].max().date()}')
print(f'Validation period: {purchase_dates[is_validation].min().date()} to {purchase_dates[is_validation].max().date()}')

Cutoff date: 2023-06-17
Training rows: 5,236
Validation rows: 1,298
Training period: 2015-07-11 to 2023-06-13
Validation period: 2023-06-17 to 2025-02-15


## 4. Baseline pipeline

For each version of the data, the preprocessing is fitted only on the training portion:

- numerical variables: median imputation and standard scaling;
- categorical variables: an explicit `Missing` level and one-hot encoding;
- identifiers (`ID`, `customer_id`) and the raw purchase date: excluded from this minimal baseline.

`customer_since_year` is treated as categorical because its relationship with the target need not be linear.

In [5]:
DROP_FROM_MODEL = [TARGET, 'ID', 'customer_id', 'purchase_date']
FORCE_CATEGORICAL = {'customer_since_year'}

def get_model_matrix(frame):
    X = frame.drop(columns=[column for column in DROP_FROM_MODEL if column in frame.columns]).copy()
    numerical = [
        column for column in X.select_dtypes(include='number').columns
        if column not in FORCE_CATEGORICAL
    ]
    categorical = [column for column in X.columns if column not in numerical]
    return X, numerical, categorical

def make_preprocessor(numerical, categorical):
    numerical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numeric', numerical_pipe, numerical),
        ('categorical', categorical_pipe, categorical),
    ])

def evaluate_logistic_regression(dataset_name, frame):
    X, numerical, categorical = get_model_matrix(frame)
    y = frame[TARGET]

    pipeline = Pipeline([
        ('preprocessor', make_preprocessor(numerical, categorical)),
        ('model', LogisticRegression(max_iter=2000, solver='liblinear', random_state=RANDOM_STATE)),
    ])
    pipeline.fit(X.loc[is_training], y.loc[is_training])
    predictions = pipeline.predict(X.loc[is_validation])
    probabilities = pipeline.predict_proba(X.loc[is_validation])[:, 1]

    return {
        'dataset': dataset_name,
        'model': 'Logistic Regression',
        'raw_predictors': X.shape[1],
        'accuracy': accuracy_score(y.loc[is_validation], predictions),
        'f1': f1_score(y.loc[is_validation], predictions),
        'roc_auc': roc_auc_score(y.loc[is_validation], probabilities),
    }


## 5. Results

We use one simple Logistic Regression model for the intermediate delivery. Accuracy and F1 describe performance at the default 0.5 classification threshold; ROC-AUC evaluates how well the model ranks future repeat purchasers independently of that threshold.

The client variables should be retained only if they produce a meaningful and repeatable improvement over the transaction-only version. One hold-out result is evidence for a next experiment, not a final claim.

In [6]:
results = pd.DataFrame([
    evaluate_logistic_regression('Transaction-only', train),
    evaluate_logistic_regression('Transaction + selected client variables', train_with_clients),
])

results = results.sort_values(['model', 'dataset']).reset_index(drop=True)
display(results.style.format({'accuracy': '{:.3f}', 'f1': '{:.3f}', 'roc_auc': '{:.3f}'}))

,dataset,model,raw_predictors,accuracy,f1,roc_auc
0,Transaction + selected client variables,Logistic Regression,41,0.696,0.722,0.781
1,Transaction-only,Logistic Regression,35,0.714,0.747,0.787


In [7]:
comparison = results.set_index('dataset')[['accuracy', 'f1', 'roc_auc']].round(3)
comparison

,accuracy,f1,roc_auc
dataset,,,
Transaction + selected client variables,0.696,0.722,0.781
Transaction-only,0.714,0.747,0.787


## 6. Interpretation and next steps

After running the notebook, write the conclusion from the displayed table. Do not claim that geographic attributes cause repeat purchases; they may simply be predictive proxies for customer profiles or commercial context.

Next controlled experiments: (1) add `city` to the selected set, (2) decide whether high-cardinality `postal_code` adds stable value, (3) test feature-selection methods inside validation, and (4) evaluate the strongest candidates with temporal cross-validation.